# Part A — Business-Level Clustering (notebook)

### Imports

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

## A1 — Data acquisition and cleaning

- Load the GeoJSON and inspect structure, types, and missingness yourself
- Decide how to handle records with no geographic coordinates, and missing `numberofemployees`/`feepaid` — justify your choice
- Filter to a sensible subset (e.g., `status == "Issued"`) and justify
businesstype has ~94 categories, many sparse — decide how to consolidate (e.g., keep the top N, group the rest as "Other")
- **Optional refinement to try:** combine `businesstype` and `businesssubtype` into a single, more fine-grained industry label where a subtype exists (falling back to `businesstype` alone where it doesn't). This gives you a sharper industry signal for some categories — worth comparing against `businesstype` alone.
- Keep your sample size manageable for both plotting and later Streamlit responsiveness — the Build Guide covers this

> **Hints:**
> - Load the GeoJSON with `geopandas.read_file()` (gives you a ready-to-use `geometry` column), or with `json.load()` + `pd.json_normalize(data["features"])` if you'd rather stick with plain pandas — either is fine.
> - The fields you actually need live under each feature's `properties` — if `pd.json_normalize` doesn't flatten them automatically, pass `record_path`/prefix arguments, or normalize `[f["properties"] for f in data["features"]]` directly.
> - `geo_point_2d`/`geometry` is what gives you usable latitude/longitude — a business without it can't be placed on a map or used in A2's location clustering, so those rows are natural candidates to drop (but say so explicitly).
> - A fast way to decide what to do with a column: `df["column"].isna().mean()`. Run it on `numberofemployees`, `feepaid`, `businesssubtype`, and `postalcode` before deciding.
> - For consolidating `businesstype`, `df["businesstype"].value_counts()` shows you the long tail directly — a defensible approach is keeping however many categories cover ~80–90% of records and bucketing the rest as "Other."
> - `status` has values beyond `Issued` (pending, cancelled, gone out of business, etc.) — `df["status"].value_counts()` shows what you're including or excluding.
> - You don't need to hand-fix every messy field (inconsistent `province` values, duplicate `licencenumber` from revisions, etc.) — a reasonable, clearly justified pass on the columns your features actually use is enough.

> **Markdown required:** Document each cleaning decision and why you made it.

## A2 — Location-only clustering: K-means vs. DBSCAN

Using **only latitude/longitude**, cluster the businesses two ways:

- **K-means:** use the elbow method to choose K, then fit
- **DBSCAN:** choose `eps`/`min_samples` reasonably (some trial and error expected)
- Produce a **static plot** (matplotlib) of the points colored by cluster assignment, one for each method

> **Markdown required:**
> 1. How do the K-means clusters map onto actual Vancouver geography — do they correspond to anything you'd recognize (downtown core, west side, etc.)?
> 2. How do the DBSCAN clusters compare? Where do the two methods agree or disagree, and why — think about what each algorithm assumes about cluster shape and density, and what DBSCAN's "noise" points represent.

## A3 — Feature-based clustering: Size, Industry, Lifecycle

Engineer and scale features from:

- **Size** — number of employees, fee paid
- **Industry** — your consolidated business type (or type+subtype) categories, encoded
- **Lifecycle** — licence duration (`expireddate − issueddate`), month issued (cyclically encoded), licence revision number
- (Feel free to include other features you think are useful)

Run a simple K-means clustering on the combined, scaled feature set, then apply PCA to project to 2D and visualize the clusters.

> **Markdown required:** What do the resulting clusters seem to represent? Does this differ from the purely geographic clustering in A2?

> **Optional/suggested (not required):** Profile each cluster using z-scores of the input features (as shown in lecture) to name each cluster — a nice way to deepen your interpretation, but not graded as a separate requirement.